# 1. Initialisation

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

llm_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

## le modèle n'a pas accès à ta base FAQ. 

In [38]:
response1 = llm_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages
)

In [39]:
response1.choices[0].message.content

'I need more information about the course. Could you please provide more details, such as the course name, institution, or any relevant deadlines? This will help me better understand your query and provide a more accurate response.'

# 2. La fonction search

In [27]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [30]:
search('Ollama: How to install Ollama??')

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

In [37]:
search("How d  I use LlLM locally with Olama?")

[{'id': 'aa310de435',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Can I run the course locally instead of Codespaces?',
  'answer': 'Yes. Codespaces is just the easiest way for everyone to start with the same environment.\n\nYou can run the course locally if you are comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module.\n\nIf you run locally, make sure you document your setup and keep your environment reproducible.'},
 {'id': 'd8b28c47a0',
  'course': 'llm-zoomcamp',
  'section': 'Capstone Project',
  'question': 'What can I use to extract text from PDFs (or scanned/OCR documents) for my RAG project?',
  'answer': 'The course doesn\'t prescribe a specific tool — use whatever works best for your data. Some options people in the community reach for:\n\n- **[docling](https://github.com/docling-project/docling)** — converts PDFs (and other docs) to structured Markdown/JSON, good for RAG.\n- **[pymupdf4llm](https://pymup

# 3. Définir le Tool

In [3]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query"
                }
            },
            "required": ["query"]
        }
    }
}

# 4. Question utilisateur

In [4]:
messages = [
    {
        "role": "system",
        "content": "You are an assistant. When a relevant tool is available, use it instead of answering from your own knowledge."
    },
    {
        "role": "user",
        "content": "I just discovered the course. Can I join it?"
    }
]

# 5. Premier appel du LLM

In [43]:
response = llm_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages,
    tools=[search_tool],
    tool_choice="auto",
    temperature=0
)

In [44]:
response

ChatCompletion(id='chatcmpl-d45276e2-cdbb-4c9a-8342-1a559eb45fda', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='xa555w29t', function=Function(arguments='{"query":"joining the course after discovery"}', name='search'), type='function')]))], created=1783301508, model='llama-3.3-70b-versatile', object='chat.completion', moderation=None, service_tier='on_demand', system_fingerprint='fp_ba38bbab80', usage=CompletionUsage(completion_tokens=17, prompt_tokens=244, total_tokens=261, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.040599605, prompt_time=0.020709338, completion_time=0.072218295, total_time=0.092927633), usage_breakdown=None, x_groq={'id': 'req_01kwtgv3c4e3ktmfcfvbt8zt6y', 'seed': 111895174})

In [42]:
response.choices[0].message.content

 # 6. Vérifier si le LLM veut appeler une fonction

In [10]:
message = response.choices[0].message

print(message.tool_calls)

[ChatCompletionMessageFunctionToolCall(id='9mqkvxzd2', function=Function(arguments='{"query":"joining the course after discovery"}', name='search'), type='function')]


In [12]:
call=response.choices[0].message

In [13]:
call

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='kmjtjv0wd', function=Function(arguments='{"query":"joining the course after discovery"}', name='search'), type='function')])

# 7. Exécuter la fonction

ouvrir la base faq.db

In [7]:
from sqlitesearch import TextSearchIndex

index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [8]:
import json

tool_call = message.tool_calls[0]

args = json.loads(tool_call.function.arguments)

results = search(**args)

result_json = json.dumps(results, indent=2)

In [45]:
args

{'query': 'joining the course after discovery'}

In [12]:
result_json 

'[\n  {\n    "id": "bd31146b0e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "When will the course be offered next?",\n    "answer": "Summer 2027."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\n    "answer": "No, you can only get a certificate if you finish the course with a \\"live\\" cohort.\\n\\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\\n\\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."\n  },\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "a

# 8. Ajouter les nouveaux messages

In [47]:
messages.append({
    "role": "assistant",
    "content": message.content,
    "tool_calls": message.tool_calls
})

messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": result_json
})

# 9. Deuxième appel

In [49]:
messages

[{'role': 'system',
  'content': 'You are an assistant. When a relevant tool is available, use it instead of answering from your own knowledge.'},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [ChatCompletionMessageFunctionToolCall(id='9mqkvxzd2', function=Function(arguments='{"query":"joining the course after discovery"}', name='search'), type='function')]},
 {'role': 'tool',
  'tool_call_id': '9mqkvxzd2',
  'content': '[\n  {\n    "id": "bd31146b0e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "When will the course be offered next?",\n    "answer": "Summer 2027."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\n    "answer": "No, you can only get a certificate if you fini

In [50]:
response = llm_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages,
    tools=[search_tool]
)

# 10. Réponse finale

In [51]:
print(response.choices[0].message.content)

Yes, you can join the course. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.


Token usage and cost

Voir le nombre de tokens utilisés

In [52]:
usage = response.usage

print("Input tokens :", usage.prompt_tokens)
print("Output tokens :", usage.completion_tokens)
print("Total tokens :", usage.total_tokens)

Input tokens : 937
Output tokens : 30
Total tokens : 967


Fonction pour calculer le prix

In [53]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

In [54]:
def calculate_groq_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.59
    OUTPUT_PRICE_PER_MILLION = 0.79

    input_cost = input_tokens / 1_000_000 * INPUT_PRICE_PER_MILLION
    output_cost = output_tokens / 1_000_000 * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }

In [55]:
usage = response.usage

result = calculate_groq_price(
    usage.prompt_tokens,
    usage.completion_tokens
)

print(result)

{'input_cost': 0.0005528299999999999, 'output_cost': 2.37e-05, 'total_cost': 0.0005765299999999999}
